In [17]:
import pandas as pd

df = pd.read_csv("../data/cleaned_transcripts.csv")
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (217, 4)


,video_id,title,datetime,transcript
0,KGVpKPNUdzA,Khabib vs Lex: Training with Khabib | FULL EXC...,2026-02-25,[Lex] All right. We're training with the Khabi...
1,YFjfBk8HI5o,OpenClaw: The Viral AI Agent that Broke the In...,2026-02-12,"- I watched my agent happily click the ""I'm no..."
2,EV7WhVT270Q,"State of AI in 2026: LLMs, Coding, Scaling Law...",2026-01-31,- The following is a conversation all about th...
3,Z-FRe5AKmCU,Paul Rosolie: Uncontacted Tribes in the Amazon...,2026-01-13,- ... were standing there. Everyone is waiting...
4,14OPT6CcsH4,"Infinity, Paradoxes, Gödel Incompleteness &amp...",2025-12-31,- The following is a conversation with Joel Da...


In [18]:
queries = []
with open("../data/search_queries.txt", "r") as f:
    for line in f:
        queries.append(line.strip())
print("Total queries:", len(queries))
print(queries[:5])

Total queries: 75
['What is machine learning?', 'How does supervised learning work?', 'What is unsupervised learning?', 'What is reinforcement learning?', 'What is overfitting in machine learning?']


In [19]:
from sentence_transformers import SentenceTransformer

model_names = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
    "multi-qa-MiniLM-L6-cos-v1"
]
models = {}
for name in model_names:
    print(f"Loading model: {name}")
    models[name] = SentenceTransformer(name)

Loading model: all-MiniLM-L6-v2
Loading model: all-mpnet-base-v2
Loading model: multi-qa-MiniLM-L6-cos-v1


In [20]:
titles = df["title"].tolist()
transcripts = df["transcript"].tolist()

title_embeddings = {}
for name, model in models.items():
    print(f"Generating title embeddings using {name}")
    embeddings = model.encode(
        titles,
        show_progress_bar=True
    )
    title_embeddings[name] = embeddings

transcript_embeddings = {}
for name, model in models.items():
    print(f"Generating transcript embeddings using {name}")
    embeddings = model.encode(
        transcripts,
        show_progress_bar=True
    )
    transcript_embeddings[name] = embeddings
    
for name in model_names:
    print(name, transcript_embeddings[name].shape)

Generating title embeddings using all-MiniLM-L6-v2


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Generating title embeddings using all-mpnet-base-v2


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Generating title embeddings using multi-qa-MiniLM-L6-cos-v1


...

Generating transcript embeddings using all-mpnet-base-v2


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Generating transcript embeddings using multi-qa-MiniLM-L6-cos-v1


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

all-MiniLM-L6-v2 (217, 384)
all-mpnet-base-v2 (217, 768)
multi-qa-MiniLM-L6-cos-v1 (217, 384)


In [21]:
query_embeddings = {}
for name, model in models.items():
    print(f"Generating query embeddings using {name}")
    embeddings = model.encode(
        queries,
        show_progress_bar=True
    )
    query_embeddings[name] = embeddings
    
for name in model_names:
    print(name, query_embeddings[name].shape)

Generating query embeddings using all-MiniLM-L6-v2


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Generating query embeddings using all-mpnet-base-v2


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Generating query embeddings using multi-qa-MiniLM-L6-cos-v1


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

all-MiniLM-L6-v2 (75, 384)
all-mpnet-base-v2 (75, 768)
multi-qa-MiniLM-L6-cos-v1 (75, 384)


In [22]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def dot_product_similarity(query_vec, doc_vecs):
    return np.dot(doc_vecs, query_vec)

def cosine_sim(query_vec, doc_vecs):
    return cosine_similarity([query_vec], doc_vecs)[0]

def rank_by_similarity(query_vec, doc_vecs, top_k=5, method="cosine"):
    if method == "cosine":
        scores = cosine_sim(query_vec, doc_vecs)
    elif method == "dot":
        scores = dot_product_similarity(query_vec, doc_vecs)
    else:
        raise ValueError("Unknown similarity method")
    
    top_indices = np.argsort(scores)[::-1][:top_k]
    return top_indices, scores[top_indices]

model_name = "all-MiniLM-L6-v2"
query_vec = query_embeddings[model_name][0]
doc_vecs = transcript_embeddings[model_name]
indices, scores = rank_by_similarity(query_vec, doc_vecs)
df.iloc[indices][["video_id", "title"]]

,video_id,title
213,fjIhFzTUB9I,"Bjarne Stroustrup: Deep Learning, Software 2.0..."
88,cdiD-9MMpb0,"Andrej Karpathy: Tesla AI, Self-Driving, Optim..."
197,1k37OcjH7BM,Andrew Ng: Advice on Getting Started in Deep L...
144,ABbDB6xri8o,Tesla AI Day Highlights | Lex Fridman
15,-HzgcbRXUK8,"Demis Hassabis: Future of AI, Simulating Reali..."


In [23]:
from scipy.spatial.distance import cdist

def rank_by_distance(query_vec, doc_vecs, top_k=5, metric="euclidean"):
    distances = cdist([query_vec], doc_vecs, metric=metric)[0]
    top_indices = np.argsort(distances)[:top_k]
    return top_indices, distances[top_indices]

metrics = [
    "euclidean",
    "cityblock",   
    "chebyshev"
]
indices, distances = rank_by_distance(
    query_vec,
    doc_vecs,
    metric="euclidean"
)
df.iloc[indices][["video_id", "title"]]

,video_id,title
213,fjIhFzTUB9I,"Bjarne Stroustrup: Deep Learning, Software 2.0..."
88,cdiD-9MMpb0,"Andrej Karpathy: Tesla AI, Self-Driving, Optim..."
197,1k37OcjH7BM,Andrew Ng: Advice on Getting Started in Deep L...
144,ABbDB6xri8o,Tesla AI Day Highlights | Lex Fridman
15,-HzgcbRXUK8,"Demis Hassabis: Future of AI, Simulating Reali..."


In [30]:
def rank_all_queries(df, model_name, top_k=5, method="cosine"):
    results = []
    doc_vecs = transcript_embeddings[model_name]
    for i, query in enumerate(queries):
        query_vec = query_embeddings[model_name][i]
        indices, scores = rank_by_similarity(
            query_vec,
            doc_vecs,
            top_k=top_k,
            method=method
        )
        
        for rank, (idx, score) in enumerate(zip(indices, scores), start=1):
            results.append({
                "query": query,
                "rank": rank,
                "video_id": df.iloc[idx]["video_id"],
                "title": df.iloc[idx]["title"],
                "score": float(score)
            })

    return pd.DataFrame(results)

model_name = "all-MiniLM-L6-v2"
results_df = rank_all_queries(
    df,
    model_name=model_name,
    top_k=5,
    method="cosine"
)
results_df.to_csv("../data/search_results.csv", index=False)
results_df.head()

,query,rank,video_id,title,score
0,What is machine learning?,1,fjIhFzTUB9I,"Bjarne Stroustrup: Deep Learning, Software 2.0...",0.539088
1,What is machine learning?,2,cdiD-9MMpb0,"Andrej Karpathy: Tesla AI, Self-Driving, Optim...",0.405098
2,What is machine learning?,3,1k37OcjH7BM,Andrew Ng: Advice on Getting Started in Deep L...,0.403038
3,What is machine learning?,4,ABbDB6xri8o,Tesla AI Day Highlights | Lex Fridman,0.389638
4,What is machine learning?,5,-HzgcbRXUK8,"Demis Hassabis: Future of AI, Simulating Reali...",0.382750


In [25]:
mapping_df = pd.read_csv("../data/query_video.csv")
mapping_df.head()

,query,relevant_video_id
0,What is machine learning?,EV7WhVT270Q
1,What is reinforcement learning?,YFjfBk8HI5o
2,How does gradient descent work?,1k37OcjH7BM
3,Explain neural networks,EV7WhVT270Q
4,What is artificial general intelligence?,EV7WhVT270Q


In [26]:
def evaluate_results(results_df, mapping_df, top_k=5):
    top1_hits = 0
    top3_hits = 0
    top5_hits = 0
    ranks = []
    total_queries = len(mapping_df)

    for _, row in mapping_df.iterrows():
        query = row["query"]
        true_video = row["relevant_video_id"]
        retrieved = results_df[results_df["query"] == query]

        retrieved_ids = retrieved["video_id"].tolist()

        if true_video in retrieved_ids[:1]:
            top1_hits += 1

        if true_video in retrieved_ids[:3]:
            top3_hits += 1

        if true_video in retrieved_ids[:5]:
            top5_hits += 1
            
        if true_video in retrieved_ids:
            rank = retrieved_ids.index(true_video) + 1
            ranks.append(rank)
        else:
            ranks.append(None)
            
    top1_recall = top1_hits / total_queries
    top3_recall = top3_hits / total_queries
    top5_recall = top5_hits / total_queries

    avg_rank = sum([r for r in ranks if r is not None]) / len([r for r in ranks if r is not None])

    return {
        "Top-1 Recall": round(top1_recall, 3),
        "Top-3 Recall": round(top3_recall, 3),
        "Top-5 Recall": round(top5_recall, 3),
        "Average Rank": round(avg_rank, 3)
    }

In [27]:
comparison_results = []

models_to_test = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
    "multi-qa-MiniLM-L6-cos-v1"
]

methods = ["cosine", "dot"]

for model_name in models_to_test:
    for method in methods:
        
        print(f"Evaluating: {model_name} + {method}")
        
        results_df = rank_all_queries(
            df,
            model_name=model_name,
            top_k=5,
            method=method
        )

        metrics = evaluate_results(results_df, mapping_df)

        comparison_results.append({
            "Model": model_name,
            "Metric": method,
            **metrics
        })
        
comparison_df = pd.DataFrame(comparison_results)
comparison_df

Evaluating: all-MiniLM-L6-v2 + cosine
Evaluating: all-MiniLM-L6-v2 + dot
Evaluating: all-mpnet-base-v2 + cosine
Evaluating: all-mpnet-base-v2 + dot
Evaluating: multi-qa-MiniLM-L6-cos-v1 + cosine
Evaluating: multi-qa-MiniLM-L6-cos-v1 + dot


,Model,Metric,Top-1 Recall,Top-3 Recall,Top-5 Recall,Average Rank
0,all-MiniLM-L6-v2,cosine,0.0,0.1,0.2,4.0
1,all-MiniLM-L6-v2,dot,0.0,0.1,0.2,4.0
2,all-mpnet-base-v2,cosine,0.1,0.1,0.2,3.0
3,all-mpnet-base-v2,dot,0.1,0.1,0.2,3.0
4,multi-qa-MiniLM-L6-cos-v1,cosine,0.1,0.1,0.1,1.0
5,multi-qa-MiniLM-L6-cos-v1,dot,0.1,0.1,0.1,1.0


In [40]:
comparison_df.to_csv("../data/model_comparison.csv", index=False)
comparison_df.sort_values(by="Top-3 Recall", ascending=False)

,Model,Metric,Top-1 Recall,Top-3 Recall,Top-5 Recall,Average Rank
0,all-MiniLM-L6-v2,cosine,0.0,0.1,0.2,4.0
1,all-MiniLM-L6-v2,dot,0.0,0.1,0.2,4.0
2,all-mpnet-base-v2,cosine,0.1,0.1,0.2,3.0
3,all-mpnet-base-v2,dot,0.1,0.1,0.2,3.0
4,multi-qa-MiniLM-L6-cos-v1,cosine,0.1,0.1,0.1,1.0
5,multi-qa-MiniLM-L6-cos-v1,dot,0.1,0.1,0.1,1.0
